In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from itertools import combinations

In [7]:


group_files = {
    # ===== HC =====
    # "HC_EMP":            "./adatas/RR/adata_HC_EMP.h5ad",
    # "HC_HSC":            "./adatas/RR/adata_HC_HSC.h5ad",
    # "HC_NK":             "./adatas/RR/adata_HC_NK.h5ad",
    # "HC_GMP1":            "./adatas/RR/adata_HC_GMP1.h5ad",
    # "HC_cDC1":           "./adatas/RR/adata_HC_cDC1.h5ad",
    # "HC_macrophage3":    "./adatas/RR/adata_HC_macrophage3.h5ad",
    # "HC_Erythroid(right)":"./adatas/RR/adata_HC_Erythroid.h5ad",

    # ===== pre_res =====
    "pre_res_MPPCLP1":   "./adatas/RR/RR_pre_res_MPPCLP1.h5ad",
    "pre_res_HSCLSC":    "./adatas/RR/RR_pre_res_HSCLSC.h5ad",
    # "pre_res_EMP":       "./adatas/RR/RR_pre_res_EMP.h5ad",
    # "pre_res_MPPCLP2":   "./adatas/RR/adata_pre_res_MPPCLP2.h5ad",
    # "pre_res_LE":        "./adatas/RR/adata_pre_res_LE.h5ad",
    # "pre_res_GMP1":      "./adatas/RR/adata_pre_res_GMP1.h5ad",
    # "pre_res_GMP2":      "./adatas/RR/adata_pre_res_GMP2.h5ad",
    # "pre_res_macrophage2":"./adatas/RR/adata_pre_res_macrophage2.h5ad",
    # "pre_res_Erythroid(right)":"./adatas/RR/adata_pre_res_Erythroid.h5ad",

    # ===== pre_nres =====
    "pre_nres_HSCLSC":   "./adatas/RR/RR_pre_nres_HSCLSC.h5ad",
    "pre_nres_MPPCLP1":  "./adatas/RR/RR_pre_nres_MPPCLP1.h5ad",
    # "pre_nres_EMP":      "./adatas/RR/RR_pre_nres_EMP.h5ad",
    # "pre_nres_LE":       "./adatas/RR/adata_pre_nres_LE.h5ad",
    # "pre_nres_Erythroid(right)":"./adatas/RR/adata_pre_nres_Erythroid.h5ad",

    # ===== post_res =====
    # "post_res_EMP":      "./adatas/RR/adata_post_res_EMP.h5ad",
    # "post_res_LE":       "./adatas/RR/adata_post_res_LE.h5ad",
    # "post_res_GMP1":     "./adatas/RR/adata_post_res_GMP1.h5ad",
    # "post_res_Erythroid(right)":"./adatas/RR/adata_post_res_Erythroid.h5ad",

    # ===== post_nres =====
    # "post_nres_ASDC":        "./adatas/RR/adata_post_nres_ASDC.h5ad",
    # "post_nres_MPPpreB":     "./adatas/RR/adata_post_nres_MPPpreB.h5ad",
    "post_nres_HSCLSC":      "./adatas/RR/RR_post_nres_HSCLSC.h5ad",
    "post_nres_MPPCLP1":      "./adatas/RR/RR_post_nres_MPPCLP1.h5ad",
    # "post_nres_EMP":         "./adatas/RR/RR_post_nres_EMP.h5ad",
    # "post_nres_Macrophage1": "./adatas/RR/adata_post_nres_Macrophage1.h5ad",
    # "post_nres_LE":          "./adatas/RR/adata_post_nres_LE.h5ad",
    # "post_nres_GMP1":        "./adatas/RR/adata_post_nres_GMP1.h5ad",
    # "post_nres_Erythroid(right)":"./adatas/RR/adata_post_nres_Erythroid.h5ad",
}

# 读入
adatas = {g: sc.read_h5ad(f) for g, f in group_files.items()}


In [9]:
for i in ['pre_res_MPPCLP1', 'pre_res_HSCLSC', 'pre_res_EMP', 'pre_nres_HSCLSC', 'pre_nres_MPPCLP1', 'pre_nres_EMP', 'post_nres_HSCLSC', 'post_nres_MPPCLP1', 'post_nres_EMP']:
    print(i)
    print(adatas[i].n_obs)
    break
    


pre_res_MPPCLP1
477


In [8]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from scipy import sparse

# =========================================================
# 0) 你的文件路径（按需修改）
# =========================================================
DATA_DIR = "./adatas/RR/"  # 放 h5ad 的文件夹，比如 "./data"

group_files = {
    # ===== LSC =====
    "pre_res_LSC":  os.path.join(DATA_DIR, "RR_pre_res_HSCLSC.h5ad"),
    "pre_nres_LSC": os.path.join(DATA_DIR, "RR_pre_nres_HSCLSC.h5ad"),

    # ===== MPP1 =====
    "pre_res_MPP1":  os.path.join(DATA_DIR, "RR_pre_res_MPPCLP1.h5ad"),
    "pre_nres_MPP1": os.path.join(DATA_DIR, "RR_pre_nres_MPPCLP1.h5ad"),

    # ===== GMP1 (用 Macrophage 作为对应) =====
    # 你给的是 pre_nres_HSCMacro vs pre_res_GMP1
    "pre_res_GMP1":  os.path.join(DATA_DIR, "RR_pre_res_GMP1.h5ad"),
    "pre_nres_GMP1": os.path.join(DATA_DIR, "RR_pre_nres_HSCMacro.h5ad"),

    # ===== EMP (Megakaryocyte) =====
    "pre_res_EMP":  os.path.join(DATA_DIR, "RR_pre_res_Megakaryocyte.h5ad"),
    "pre_nres_EMP": os.path.join(DATA_DIR, "RR_pre_nres_Megakaryocyte.h5ad"),
}

# 你要比较的 clusters（最终会分别输出）
# 每个条目：(cluster_name, res_key, nres_key)
clusters_to_test = [
    ("LSC", "pre_res_LSC", "pre_nres_LSC"),
    ("MPP1", "pre_res_MPP1", "pre_nres_MPP1"),
    ("GMP1(Macro)", "pre_res_GMP1", "pre_nres_GMP1"),
    ("EMP(Megakaryocyte)", "pre_res_EMP", "pre_nres_EMP"),
]

# =========================================================
# 1) 参数（跟你之前逻辑一致）
# =========================================================
USE_LAYER = None       # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False        # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0        # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20     # 每组至少多少表达细胞才纳入
USE_FDR = True         # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05
FC_METHOD = "mean"     # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0   # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_pre_nres_vs_pre_res_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # res
        x2_pos = x2_all[x2_all > threshold]  # nres

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_res  = summary_stat(x1_pos, method=fc_method)
        stat_nres = summary_stat(x2_pos, method=fc_method)

        denom = stat_res + fc_pseudocount
        numer = stat_nres + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_res),
            f"{fc_method}_g2_pos": float(stat_nres),
            "FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            "log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            "median_pos_diffg1-g2)": float(np.median(x1_pos) - np.median(x2_pos)),
            "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm（比如 Scanpy 标准流程），就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 cluster 跑 DE（pre_nres vs pre_res）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # cluster -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # cluster -> set(sig_up_genes)

for cluster_name, res_key, nres_key in clusters_to_test:
    ad_res = maybe_preprocess(adatas[res_key])
    ad_nres = maybe_preprocess(adatas[nres_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_res, ad_nres,
        res_name=res_key, nres_name=nres_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {cluster_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(nres>res)
    if USE_FDR:
        sig_mask = df["FDR_BH"].notna() & (df["FDR_BH"] < FDR_CUTOFF)
    else:
        sig_mask = df["p_value"].notna() & (df["p_value"] < P_CUTOFF)

    up_mask = df["log2FC_nres_over_res"].notna() & (df["log2FC_nres_over_res"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{cluster_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{cluster_name}__SIG_up_in_pre_nres.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{cluster_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["cluster"] = cluster_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[cluster_name] = df.set_index("gene")["log2FC_nres_over_res"]
    sig_gene_sets[cluster_name] = set(sig_up["gene"].tolist())

# 合并输出：所有 cluster 的显著上调基因清单（你要的 “comprehensively list”）
if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(os.path.join(OUTDIR, "ALL_clusters__SIG_up_in_pre_nres__combined.csv"), index=False)

    # 也给一个更“列表式”的版本：每个 cluster 一行，genes 用分号拼起来
    list_rows = []
    for cluster_name, genes in sig_gene_sets.items():
        list_rows.append({
            "cluster": cluster_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_pre_nres": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(
        os.path.join(OUTDIR, "ALL_clusters__SIG_up_in_pre_nres__gene_list_per_cluster.csv"),
        index=False
    )

# =========================================================
# 5) 画图：跨所有 cluster，突出 LSC 中 FC>1.3 的基因
# =========================================================
# 取 LSC 中：显著上调 且 FC>1.3
LSC_NAME = "LSC"
if LSC_NAME in log2fc_by_cluster:
    # 先拿 LSC 的“显著上调表”文件读回来（最稳）
    lsc_sig_path = os.path.join(OUTDIR, f"{LSC_NAME}__SIG_up_in_pre_nres.csv")
    if os.path.exists(lsc_sig_path):
        lsc_sig = pd.read_csv(lsc_sig_path)
        lsc_sig_fc13 = lsc_sig[lsc_sig["FC_nres_over_res"].notna() & (lsc_sig["FC_nres_over_res"] > 1.3)].copy()

        genes_focus = lsc_sig_fc13["gene"].dropna().unique().tolist()
        print(f"[Plot] LSC significant & FC>1.3 genes: {len(genes_focus)}")

        if len(genes_focus) > 0:
            # 组一个矩阵：rows=genes_focus, cols=clusters, values=log2FC
            clusters_order = [c[0] for c in clusters_to_test if c[0] in log2fc_by_cluster]
            mat = []
            for g in genes_focus:
                row = []
                for c in clusters_order:
                    s = log2fc_by_cluster[c]
                    row.append(float(s.get(g, np.nan)))
                mat.append(row)
            mat = np.array(mat, dtype=float)

            # 简单排序：按 LSC 的 log2FC 从大到小
            if LSC_NAME in clusters_order:
                lsc_col = clusters_order.index(LSC_NAME)
                order = np.argsort(-np.nan_to_num(mat[:, lsc_col], nan=-1e9))
                mat = mat[order, :]
                genes_focus = [genes_focus[i] for i in order]

            plt.figure(figsize=(1.2*len(clusters_order) + 4, 0.18*len(genes_focus) + 4))
            im = plt.imshow(mat, aspect="auto")  # 默认 colormap
            plt.colorbar(im, label="log2FC (pre_nres / pre_res)")

            plt.xticks(range(len(clusters_order)), clusters_order, rotation=30, ha="right")
            plt.yticks(range(len(genes_focus)), genes_focus, fontsize=7)

            plt.title("Genes significant in LSC with FC>1.3: log2FC across clusters")
            plt.tight_layout()

            fig_path = os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters.png")
            plt.savefig(fig_path, dpi=300)
            plt.close()
            print(f"[Plot] saved: {fig_path}")

            # 也输出该图对应矩阵
            heat_df = pd.DataFrame(mat, index=genes_focus, columns=clusters_order)
            heat_df.to_csv(os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters__log2FC_matrix.csv"))
else:
    print("[WARN] LSC not found in results; skip heatmap.")

print("Done.")


FileNotFoundError: Missing file for pre_res_LSC: ./adatas/RR/RR_pre_res_HSCLSC.h5ad

In [9]:
pwd

'd:\\Projects\\Geneformer'

'd:\\Projects\\Geneformer\\adatas\\RRErythroid'

In [13]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from scipy import sparse

# =========================================================
# 0) 你的文件路径（按需修改）
# =========================================================
DATA_DIR = "./adatas/RRErythroid"  # 放 h5ad 的文件夹，比如 "./data"

group_files = {
    # ================= pre =================
    "pre_res_Erythroid": os.path.join(DATA_DIR, "RR_pre_res_Erythroid.h5ad"),
    "pre_nres_Erythroid": os.path.join(DATA_DIR, "RR_pre_nres_Erythroid.h5ad"),

    "pre_res_LE": os.path.join(DATA_DIR, "RR_pre_res_LEunknown.h5ad"),
    "pre_nres_LE": os.path.join(DATA_DIR, "RR_pre_nres_LEcDC2.h5ad"),

    # ================= post =================
    "post_res_Erythroid": os.path.join(DATA_DIR, "RR_post_res_Erythroid.h5ad"),
    "post_nres_Erythroid": os.path.join(DATA_DIR, "RR_post_nres_Erythroid.h5ad"),

    "post_res_LE": os.path.join(DATA_DIR, "RR_post_res_LE.h5ad"),
    "post_nres_LE": os.path.join(DATA_DIR, "RR_post_nres_LE.h5ad"),
}
# 你要比较的 clusters（最终会分别输出）
# 每个条目：(cluster_name, res_key, nres_key)
clusters_to_test = [
    ("pre_res",  "pre_res_Erythroid",  "pre_res_LE"),
    ("pre_nres", "pre_nres_Erythroid", "pre_nres_LE"),
    ("post_res", "post_res_Erythroid", "post_res_LE"),
    ("post_nres","post_nres_Erythroid","post_nres_LE"),
]
# clusters_to_test = [
#     ("pre_res",  "pre_res_LE",  "pre_res_Erythroid"),
#     ("pre_nres", "pre_nres_LE", "pre_nres_Erythroid"),
#     ("post_res", "post_res_LE", "post_res_Erythroid"),
#     ("post_nres","post_nres_LE","post_nres_Erythroid"),
# ]

# =========================================================
# 1) 参数（跟你之前逻辑一致）
# =========================================================
USE_LAYER = None       # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False        # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0        # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20     # 每组至少多少表达细胞才纳入
USE_FDR = True         # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05
FC_METHOD = "mean"     # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0   # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_pre_nres_vs_pre_res_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # res
        x2_pos = x2_all[x2_all > threshold]  # nres

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_res  = summary_stat(x1_pos, method=fc_method)
        stat_nres = summary_stat(x2_pos, method=fc_method)

        denom = stat_res + fc_pseudocount
        numer = stat_nres + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_res),
            f"{fc_method}_g2_pos": float(stat_nres),
            f"FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            f"log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            # f"median_pos_diff({nres_name}-{res_name})": float(np.median(x1_pos) - np.median(x2_pos)),
            f"mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm（比如 Scanpy 标准流程），就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 cluster 跑 DE（pre_nres vs pre_res）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # cluster -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # cluster -> set(sig_up_genes)

for cluster_name, res_key, nres_key in clusters_to_test:
    ad_res = maybe_preprocess(adatas[res_key])
    ad_nres = maybe_preprocess(adatas[nres_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_res, ad_nres,
        res_name=res_key, nres_name=nres_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {cluster_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(nres>res)
    if USE_FDR:
        sig_mask = df["FDR_BH"].notna() & (df["FDR_BH"] < FDR_CUTOFF)
    else:
        sig_mask = df["p_value"].notna() & (df["p_value"] < P_CUTOFF)

    up_mask = df[f"log2FC_g2_over_g1"].notna() & (df[f"log2FC_g2_over_g1"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{cluster_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{cluster_name}__SIG_up_in_{nres_key}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{cluster_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["cluster"] = cluster_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[cluster_name] = df.set_index("gene")[f"log2FC_g2_over_g1"]
    sig_gene_sets[cluster_name] = set(sig_up["gene"].tolist())

# 合并输出：所有 cluster 的显著上调基因清单（你要的 “comprehensively list”）
if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{nres_key}__combined.csv"), index=False)

    # 也给一个更“列表式”的版本：每个 cluster 一行，genes 用分号拼起来
    list_rows = []
    for cluster_name, genes in sig_gene_sets.items():
        list_rows.append({
            "cluster": cluster_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_pre_nres": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(
        os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{nres_key}__gene_list_per_cluster.csv"),
        index=False
    )

# # =========================================================
# # 5) 画图：跨所有 cluster，突出 LSC 中 FC>1.3 的基因
# # =========================================================
# # 取 LSC 中：显著上调 且 FC>1.3
# LSC_NAME = "LSC"
# if LSC_NAME in log2fc_by_cluster:
#     # 先拿 LSC 的“显著上调表”文件读回来（最稳）
#     lsc_sig_path = os.path.join(OUTDIR, f"{LSC_NAME}__SIG_up_in_pre_nres.csv")
#     if os.path.exists(lsc_sig_path):
#         lsc_sig = pd.read_csv(lsc_sig_path)
#         lsc_sig_fc13 = lsc_sig[lsc_sig["FC_nres_over_res"].notna() & (lsc_sig["FC_nres_over_res"] > 1.3)].copy()

#         genes_focus = lsc_sig_fc13["gene"].dropna().unique().tolist()
#         print(f"[Plot] LSC significant & FC>1.3 genes: {len(genes_focus)}")

#         if len(genes_focus) > 0:
#             # 组一个矩阵：rows=genes_focus, cols=clusters, values=log2FC
#             clusters_order = [c[0] for c in clusters_to_test if c[0] in log2fc_by_cluster]
#             mat = []
#             for g in genes_focus:
#                 row = []
#                 for c in clusters_order:
#                     s = log2fc_by_cluster[c]
#                     row.append(float(s.get(g, np.nan)))
#                 mat.append(row)
#             mat = np.array(mat, dtype=float)

#             # 简单排序：按 LSC 的 log2FC 从大到小
#             if LSC_NAME in clusters_order:
#                 lsc_col = clusters_order.index(LSC_NAME)
#                 order = np.argsort(-np.nan_to_num(mat[:, lsc_col], nan=-1e9))
#                 mat = mat[order, :]
#                 genes_focus = [genes_focus[i] for i in order]

#             plt.figure(figsize=(1.2*len(clusters_order) + 4, 0.18*len(genes_focus) + 4))
#             im = plt.imshow(mat, aspect="auto")  # 默认 colormap
#             plt.colorbar(im, label="log2FC (pre_nres / pre_res)")

#             plt.xticks(range(len(clusters_order)), clusters_order, rotation=30, ha="right")
#             plt.yticks(range(len(genes_focus)), genes_focus, fontsize=7)

#             plt.title("Genes significant in LSC with FC>1.3: log2FC across clusters")
#             plt.tight_layout()

#             fig_path = os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters.png")
#             plt.savefig(fig_path, dpi=300)
#             plt.close()
#             print(f"[Plot] saved: {fig_path}")

#             # 也输出该图对应矩阵
#             heat_df = pd.DataFrame(mat, index=genes_focus, columns=clusters_order)
#             heat_df.to_csv(os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters__log2FC_matrix.csv"))
# else:
#     print("[WARN] LSC not found in results; skip heatmap.")

print("Done.")


[pre_res] ok_genes=2410  sig_up=654
  saved: DE_pre_nres_vs_pre_res_by_cluster\pre_res__SIG_up_in_pre_res_LE.csv
[pre_nres] ok_genes=4087  sig_up=191
  saved: DE_pre_nres_vs_pre_res_by_cluster\pre_nres__SIG_up_in_pre_nres_LE.csv
[post_res] ok_genes=3994  sig_up=258
  saved: DE_pre_nres_vs_pre_res_by_cluster\post_res__SIG_up_in_post_res_LE.csv
[post_nres] ok_genes=4745  sig_up=3192
  saved: DE_pre_nres_vs_pre_res_by_cluster\post_nres__SIG_up_in_post_nres_LE.csv
Done.


In [15]:


# ===== 1. 读入数据 =====
# df = pd.read_csv("../../Gao/RR/Upregulated in Post-nonresponder compared to pre-nonresponder0.csv")
df = pd.read_csv("./adatas/RRErythroid/DE_pre_nres_vs_pre_res_by_cluster/ALL_clusters__SIG_up_in_AML_Erythroid__combined.csv")

# 如果列名和你实际的不完全一致，可以在这里统一
df = df.rename(columns={
    "Gene": "gene",          # 如果原来是 Gene
    "cluster": "pair",      # 如果原来不是 pair
})

# ===== 2. 只保留你关心的 pair（可选，但推荐）=====
pairs_keep = ["pre_res", "pre_nres", "post_res",'post_nres']
df = df[df["pair"].isin(pairs_keep)]

# ===== 3. 设置多级索引：gene × pair =====
df_idx = df.set_index(["gene", "pair"])

# ===== 4. 选择要展开的指标列 =====
# value_cols = [
#     "p_value",
#     "mean_pre_pos",
#     "mean_post_pos",
#     "FC_post_over_pre"
# ]
value_cols = [
    "p_value",
    "mean_g1_pos",
    "mean_g2_pos",
    "FC_g2_over_g1"
]
# ===== 5. unstack pair → 自动补 NaN =====
df_wide = (
    df_idx[value_cols]
    .unstack("pair")        # columns 变成 (metric, pair)
)

# ===== 6. 调整列顺序：pair 在前，metric 在后 =====
df_wide = df_wide.swaplevel(0, 1, axis=1)
df_wide = df_wide.sort_index(axis=1, level=0)

# ===== 7. 压平成单层列名（推荐）=====
df_wide.columns = [
    f"{pair}__{metric}"
    for pair, metric in df_wide.columns
]
df_wide.to_csv('Upregulated in AML Erythroid compared to normal Erythroid.csv')

In [ ]:
# import scanpy as sc
# import numpy as np
# import pandas as pd
# from scipy.stats import mannwhitneyu
# from scipy import sparse

# # =========================
# # 0) 你的数据（已读入可跳过）
# # =========================
# # group_files = {...}
# # adatas = {g: sc.read_h5ad(f) for g, f in group_files.items()}

# # =========================
# # 1) 你要比较的 pairs（默认 g1=res, g2=nres）
# # =========================
# pairs = [
#     ("pre_nres_HSCLSC",   "post_nres_HSCLSC"),
#     # ("pre_nres_EMP",      "post_nres_EMP"),
#     ("pre_nres_MPPCLP1",  "post_nres_MPPCLP1"),
# ]

# # =========================
# # 2) 参数
# # =========================
# USE_LAYER = None     # 例如 "counts" 或 "lognorm"；不用就 None
# USE_RAW = False      # 若要用 adata.raw，就 True（优先于 layers）
# THRESHOLD = 0.0      # 只用 >threshold 的表达细胞做检验/FC
# MIN_POS_CELLS = 20   # 每组至少多少表达细胞才纳入（不足则直接丢弃，不输出）

# # 显著性筛选：用 p-value 或 FDR_BH 二选一
# USE_FDR = False
# P_CUTOFF = 0.05
# FDR_CUTOFF = 0.05

# # Fold change 计算方式：mean 或 median（二选一）
# FC_METHOD = "mean"   # "mean" 或 "median"
# FC_PSEUDOCOUNT = 0.0 # 如果你担心分母为0，可设一个小值如 1e-9；但positive-only通常mean>0

# # 输出文件
# OUT_FULL = "conditional_positive_only_ALLGENES_full_results_okonly.csv"
# OUT_SIG  = "conditional_positive_only_ALLGENES_sig_up_in_nres.csv"

# # =========================
# # 3) 工具函数
# # =========================
# def bh_fdr(pvals):
#     """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
#     pvals = np.asarray(pvals, dtype=float)
#     qvals = np.full_like(pvals, np.nan, dtype=float)
#     ok = np.isfinite(pvals)
#     if ok.sum() == 0:
#         return qvals
#     pv = pvals[ok]
#     order = np.argsort(pv)
#     ranked = pv[order]
#     m = len(ranked)
#     q = ranked * m / (np.arange(1, m + 1))
#     q = np.minimum.accumulate(q[::-1])[::-1]
#     out = np.empty_like(q)
#     out[order] = q
#     qvals[ok] = out
#     return qvals

# def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
#     """
#     返回 (X, gene_names)
#     X: cells x genes (csr_matrix or ndarray)
#     gene_names: np.array of gene symbols
#     """
#     if use_raw:
#         if adata.raw is None:
#             return None, None
#         X = adata.raw.X
#         genes = np.asarray(adata.raw.var_names)
#     elif use_layer is not None:
#         if use_layer not in adata.layers:
#             return None, None
#         X = adata.layers[use_layer]
#         genes = np.asarray(adata.var_names)
#     else:
#         X = adata.X
#         genes = np.asarray(adata.var_names)

#     if sparse.issparse(X):
#         X = X.tocsr()
#     else:
#         X = np.asarray(X)
#     return X, genes

# def summary_stat(x, method="mean"):
#     if method == "median":
#         return float(np.median(x))
#     return float(np.mean(x))

# def conditional_mw_positive_only_allgenes_okonly(
#     ad1, ad2, g1_name, g2_name,
#     min_pos_cells=20, threshold=0.0,
#     use_layer=None, use_raw=False,
#     fc_method="mean",
#     fc_pseudocount=0.0
# ):
#     """
#     对所有共有基因做 positive-only MWU：
#       - 仅使用 >threshold 的细胞
#       - 若任一组 positive cells < min_pos_cells：直接丢弃（不输出任何 status 行）
#       - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
#     返回：每个基因一行（只包含满足 min_pos_cells 的基因）
#     """
#     X1, genes1 = get_matrix_and_genes(ad1, use_layer=use_layer, use_raw=use_raw)
#     X2, genes2 = get_matrix_and_genes(ad2, use_layer=use_layer, use_raw=use_raw)
#     if X1 is None or X2 is None:
#         return pd.DataFrame()

#     common = np.intersect1d(genes1, genes2, assume_unique=False)
#     if common.size == 0:
#         return pd.DataFrame()

#     idx1 = pd.Index(genes1).get_indexer(common)
#     idx2 = pd.Index(genes2).get_indexer(common)

#     X1c = X1[:, idx1]
#     X2c = X2[:, idx2]

#     rows = []
#     for j, gene in enumerate(common):
#         # 取列向量 -> dense 1D
#         if sparse.issparse(X1c):
#             x1_all = X1c[:, j].toarray().ravel()
#             x2_all = X2c[:, j].toarray().ravel()
#         else:
#             x1_all = np.asarray(X1c[:, j]).ravel()
#             x2_all = np.asarray(X2c[:, j]).ravel()

#         # positive-only
#         x1_pos = x1_all[x1_all > threshold]  # g1 (res)
#         x2_pos = x2_all[x2_all > threshold]  # g2 (nres)

#         if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
#             continue  # ✅ 丢弃，不输出 too_few_positive_cells

#         # MWU
#         u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

#         # FC / log2FC: nres/res（基于 positive-only）
#         stat_res  = summary_stat(x1_pos, method=fc_method)
#         stat_nres = summary_stat(x2_pos, method=fc_method)

#         denom = stat_res + fc_pseudocount
#         numer = stat_nres + fc_pseudocount
#         fc = (numer / denom) if denom > 0 else np.nan
#         log2fc = np.log2(fc) if (fc is not None and np.isfinite(fc) and fc > 0) else np.nan

#         rows.append({
#             "pair": f"{g1_name} vs {g2_name}",
#             "gene": gene,
#             "threshold": float(threshold),

#             "p_value": float(p),
#             "U": float(u),

#             # direction + effect size
#             f"{g1_name}_n_pos": int(x1_pos.size),
#             f"{g2_name}_n_pos": int(x2_pos.size),

#             f"{fc_method}_res_pos": float(stat_res),
#             f"{fc_method}_nres_pos": float(stat_nres),
#             "FC_nres_over_res": float(fc) if np.isfinite(fc) else np.nan,
#             "log2FC_nres_over_res": float(log2fc) if np.isfinite(log2fc) else np.nan,

#             # 额外：保留你之前的差值指标（g1-g2；nres上调则为负）
#             "median_pos_diff(g1-g2)": float(np.median(x1_pos) - np.median(x2_pos)),
#             "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
#         })

#     return pd.DataFrame(rows)

# # =========================
# # 4) 主流程：对每个 pair 跑全基因（只保留 ok-only）
# # =========================
# all_res = []
# for g1, g2 in pairs:
#     if g1 not in adatas or g2 not in adatas:
#         raise KeyError(f"Missing group in adatas: {g1} or {g2}")

#     df_pair = conditional_mw_positive_only_allgenes_okonly(
#         adatas[g1], adatas[g2],
#         g1, g2,
#         min_pos_cells=MIN_POS_CELLS,
#         threshold=THRESHOLD,
#         use_layer=USE_LAYER,
#         use_raw=USE_RAW,
#         fc_method=FC_METHOD,
#         fc_pseudocount=FC_PSEUDOCOUNT
#     )
#     all_res.append(df_pair)

# res_df = pd.concat(all_res, ignore_index=True)

# # 如果某些 pair 全被过滤掉，避免后续报错
# if res_df.shape[0] == 0:
#     print("No genes passed MIN_POS_CELLS filter across all pairs.")
#     res_df.to_csv(OUT_FULL, index=False)
#     raise SystemExit

# # =========================
# # 5) FDR（对当前保留下来的所有检验做 BH）
# # =========================
# res_df["FDR_BH"] = bh_fdr(res_df["p_value"].values)

# # =========================
# # 6) 筛选：显著 + nres 上调（log2FC>0）
# # =========================
# if USE_FDR:
#     sig_mask = res_df["FDR_BH"].notna() & (res_df["FDR_BH"] < FDR_CUTOFF)
# else:
#     sig_mask = res_df["p_value"].notna() & (res_df["p_value"] < P_CUTOFF)

# up_in_nres_mask = res_df["log2FC_nres_over_res"].notna() & (res_df["log2FC_nres_over_res"] > 0)

# sig_up_df = res_df[sig_mask & up_in_nres_mask].copy()

# # 排序输出（更友好）
# sort_cols = ["pair", "FDR_BH"] if USE_FDR else ["pair", "p_value"]
# sig_up_df = sig_up_df.sort_values(sort_cols, ascending=True)

# print("Rows kept after MIN_POS_CELLS filtering:", res_df.shape[0])
# print("Significant & up in nres:", sig_up_df.shape[0])

# show_cols = [
#     "pair", "gene",
#     "p_value", "FDR_BH",
#     "log2FC_nres_over_res", "FC_nres_over_res",
#     f"{FC_METHOD}_res_pos", f"{FC_METHOD}_nres_pos",
#     "median_pos_diff(g1-g2)", "mean_pos_diff(g1-g2)",
# ]
# show_cols = [c for c in show_cols if c in sig_up_df.columns]
# print(sig_up_df[show_cols].head(50).to_string(index=False))

# # =========================
# # 7) 保存
# # =========================
# res_df.to_csv(OUT_FULL, index=False)
# sig_up_df.to_csv(OUT_SIG, index=False)
# print(f"Saved full results: {OUT_FULL}")
# print(f"Saved significant up-in-nres: {OUT_SIG}")


Rows kept after MIN_POS_CELLS filtering: 8865
Significant & up in nres: 327
                               pair     gene      p_value       FDR_BH  log2FC_nres_over_res  FC_nres_over_res  mean_res_pos  mean_nres_pos  median_pos_diff(g1-g2)  mean_pos_diff(g1-g2)
pre_nres_HSCLSC vs post_nres_HSCLSC   EEF1A1 5.468032e-69 4.847410e-65              1.257187          2.390292      1.826691       4.366324               -2.743922             -2.539633
pre_nres_HSCLSC vs post_nres_HSCLSC     PTMA 2.383880e-65 1.056655e-61              0.921029          1.893466      1.741273       3.297040               -1.651876             -1.555767
pre_nres_HSCLSC vs post_nres_HSCLSC     TPT1 1.896916e-60 5.605387e-57              1.309276          2.478172      1.723944       4.272230               -2.849206             -2.548286
pre_nres_HSCLSC vs post_nres_HSCLSC      B2M 6.399516e-58 1.418293e-54              0.961156          1.946869      1.736356       3.380458               -1.731186             -1.6

In [24]:


# ===== 1. 读入数据 =====
# df = pd.read_csv("../../Gao/RR/Upregulated in Post-nonresponder compared to pre-nonresponder0.csv")
df = pd.read_csv("./DE_pre_nres_vs_pre_res_by_cluster/ALL_clusters__SIG_up_in_pre_nres__combined.csv")

# 如果列名和你实际的不完全一致，可以在这里统一
df = df.rename(columns={
    "Gene": "gene",          # 如果原来是 Gene
    "cluster": "pair",      # 如果原来不是 pair
})

# ===== 2. 只保留你关心的 pair（可选，但推荐）=====
pairs_keep = ["EMP(Megakaryocyte)", "MPP1", "LSC",'GMP1(Macro)']
df = df[df["pair"].isin(pairs_keep)]

# ===== 3. 设置多级索引：gene × pair =====
df_idx = df.set_index(["gene", "pair"])

# ===== 4. 选择要展开的指标列 =====
# value_cols = [
#     "p_value",
#     "mean_pre_pos",
#     "mean_post_pos",
#     "FC_post_over_pre"
# ]
value_cols = [
    "p_value",
    "mean_res_pos",
    "mean_nres_pos",
    "FC_nres_over_res"
]
# ===== 5. unstack pair → 自动补 NaN =====
df_wide = (
    df_idx[value_cols]
    .unstack("pair")        # columns 变成 (metric, pair)
)

# ===== 6. 调整列顺序：pair 在前，metric 在后 =====
df_wide = df_wide.swaplevel(0, 1, axis=1)
df_wide = df_wide.sort_index(axis=1, level=0)

# ===== 7. 压平成单层列名（推荐）=====
df_wide.columns = [
    f"{pair}__{metric}"
    for pair, metric in df_wide.columns
]
df_wide.to_csv('Upregulated in Pre-nonresponder compared to pre-responder.csv')

In [29]:
df_wide[df_wide['LSC__FC_nres_over_res']>1.3].to_csv('Upregulated in Pre-nonresponder compared to pre-responder_FCgt1p3.csv')

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

files = {
    "HC_HSC": "./adatas/RR/Comparison/RR_HC_HSC.h5ad",
    "HC_Megakaryocyte": "./adatas/RR/Comparison/RR_HC_Megakaryocyte.h5ad",

    "pre_res_HSCLSC": "./adatas/RR/Comparison/RR_pre_res_HSCLSC.h5ad",
    "pre_nres_HSCLSC": "./adatas/RR/Comparison/RR_pre_nres_HSCLSC.h5ad",
    "post_nres_HSCLSC": "./adatas/RR/Comparison/RR_post_nres_HSCLSC.h5ad",
    # "pre_nres_HSCMacro": "./adatas/RR/RR_pre_nres_HSCMacro.h5ad",

    "pre_res_MPPPreB": "./adatas/RR/Comparison/RR_pre_res_MPPPreB.h5ad",
    "pre_nres_MPPPreB": "./adatas/RR/Comparison/RR_pre_nres_MPPPreB.h5ad",

    "pre_res_MPPCLP1": "./adatas/RR/Comparison/RR_pre_res_MPPCLP1.h5ad",
    "pre_nres_MPPCLP1": "./adatas/RR/Comparison/RR_pre_nres_MPPCLP1.h5ad",
    "post_nres_MPPCLP1": "./adatas/RR/Comparison/RR_post_nres_MPPCLP1.h5ad",
    "post_nres_ASDC": "./adatas/RR/Comparison/RR_post_nres_ASDC.h5ad",
    "post_nres_MPPpreB": "./adatas/RR/Comparison/RR_post_nres_MPPpreB.h5ad",

    "pre_res_Megakaryocyte": "./adatas/RR/Comparison/RR_pre_res_Megakaryocyte.h5ad",
    "pre_nres_Megakaryocyte": "./adatas/RR/Comparison/RR_pre_nres_Megakaryocyte.h5ad",
    "post_nres_Megakaryocyte": "./adatas/RR/Comparison/RR_post_nres_Megakaryocyte.h5ad",
    "post_nres_progenitor": "./adatas/RR/Comparison/RR_post_nres_progenitor.h5ad",
}

adatas = {k: sc.read_h5ad(v) for k, v in files.items()}

genes = [
    "ASXL1",
    "BAALC",
    "BCL11A",
    "CALCRL",
    "ELMO1",
    "ERG",
    "FOXN3",
    "GNAI1",
    "LDLRAD4",
    "LRBA",
    "MLLT3",
    "NCOA7",
    "NIBAN1",
    "NOL4L",
    "NPR3",
    "NR6A1",
    "NRIP1",
    "PDZD8",
    "PRKACB",
    "RBPMS",
    "RIPOR2",
    "TCF12",
    "TNS3",
    "VAV3",
    "ZNF718",
]



In [ ]:
adatas

In [2]:
def mean_expression(adata, genes):
    present = [g for g in genes if g in adata.var_names]
    X = adata[:, present].X

    if sparse.issparse(X):
        mean_vals = np.asarray(X.mean(axis=0)).ravel()
    else:
        mean_vals = X.mean(axis=0)

    return pd.Series(mean_vals, index=present)
mean_expr = {}
for name, ad in adatas.items():
    mean_expr[name] = mean_expression(ad, genes)


In [3]:
comparisons = {
    "HSCLSC": {
        "baseline": "pre_res_HSCLSC",
        "targets": [
            "HC_HSC",
            "pre_nres_HSCLSC",
            "post_nres_HSCLSC",
            # "pre_nres_HSCMacro",
        ],
    },
    # "GMP1": {
    #     "baseline": "pre_res_GMP1",
    #     "targets": ["post_nres_GMP1"],
    # },
    "MPPCLP1": {
        "baseline": "pre_res_MPPPreB",
        "targets": [
            "pre_res_MPPCLP1",
            "pre_nres_MPPCLP1",
            "pre_nres_MPPPreB",
            "post_nres_MPPCLP1",
            "post_nres_ASDC",
            "post_nres_MPPpreB",
        ],
    },
    "Megakaryocyte": {
        "baseline": "post_nres_progenitor",
        "targets": [
            "pre_res_Megakaryocyte",
            "HC_Megakaryocyte",
            "pre_nres_Megakaryocyte",
            "post_nres_Megakaryocyte",
        ],
    },
}


In [4]:
rows = []

for group, cfg in comparisons.items():
    base = cfg["baseline"]
    base_mean = mean_expr[base]

    for tgt in cfg["targets"]:
        tgt_mean = mean_expr[tgt]

        common = base_mean.index.intersection(tgt_mean.index)
        ratio = tgt_mean[common] / base_mean[common]

        for g in common:
            rows.append({
                "group": group,
                "baseline": base,
                "target": tgt,
                "gene": g,
                "mean_baseline": base_mean[g],
                "mean_target": tgt_mean[g],
                "ratio_target_over_baseline": ratio[g],
                "log2_ratio": np.log2(ratio[g]) if ratio[g] > 0 else np.nan
            })

ratio_df = pd.DataFrame(rows)


In [5]:
ratio_df.to_csv(
    "gene_mean_expression_ratio_vs_baseline_all_groups.csv",
    index=False
)

print("Saved: gene_mean_expression_ratio_vs_baseline_all_groups.csv")


Saved: gene_mean_expression_ratio_vs_baseline_all_groups.csv


In [6]:
ratio_df["comparison"] = (
    ratio_df["group"] + "_" + ratio_df["target"]
)


In [39]:
ratio_matrix = ratio_df.pivot(
    index="gene",
    columns="comparison",
    values="ratio_target_over_baseline"
)


In [40]:
ratio_matrix.to_csv(
    "gene_mean_expression_ratio__GENE_ROWS.csv"
)

print("Saved wide gene-by-comparison matrices.")


Saved wide gene-by-comparison matrices.


In [ ]:

# ===== 你要比较的 pairs & genes =====
pairs = [
    ("pre_res_HSCLSC",   "pre_nres_HSCLSC"),
    ("pre_res_EMP",      "pre_nres_EMP"),
    ("pre_res_MPPCLP1",  "pre_nres_MPPCLP1"),
]


# ===== 工具函数 =====
def get_gene_vector(adata, gene, use_layer=None, use_raw=False):
    """
    从 adata 中提取 gene 的表达向量（dense 1D np.array）
    - use_raw=True: 从 adata.raw 取
    - use_layer="counts"/"lognorm": 从 adata.layers 取
    - 默认：从 adata.X 取
    """
    if use_raw:
        if adata.raw is None:
            return None
        if gene not in adata.raw.var_names:
            return None
        x = adata.raw[:, gene].X
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None
        if gene not in adata.var_names:
            return None
        x = adata[:, gene].layers[use_layer]
    else:
        if gene not in adata.var_names:
            return None
        x = adata[:, gene].X

    if hasattr(x, "toarray"):
        x = x.toarray()
    x = np.asarray(x).reshape(-1)
    return x

def conditional_mw_positive_only(ad1, ad2, g1_name, g2_name, gene,
                                min_pos_cells=20, threshold=0.0,
                                use_layer=None, use_raw=False):
    """
    仅在表达细胞中比较（X > threshold）：
    - 先过滤掉 0（或 <=threshold）的细胞
    - 再做 Mann–Whitney U（两侧）
    同时返回 prevalence（表达比例）用于解释
    """
    x1_all = get_gene_vector(ad1, gene, use_layer=use_layer, use_raw=use_raw)
    x2_all = get_gene_vector(ad2, gene, use_layer=use_layer, use_raw=use_raw)

    if x1_all is None or x2_all is None:
        return {"pair": f"{g1_name} vs {g2_name}", "gene": gene, "status": "gene_not_found_or_layer_missing"}

    x1_pos = x1_all[x1_all > threshold]
    x2_pos = x2_all[x2_all > threshold]

    prev1 = float(np.mean(x1_all > threshold))
    prev2 = float(np.mean(x2_all > threshold))

    if len(x1_pos) < min_pos_cells or len(x2_pos) < min_pos_cells:
        return {
            "pair": f"{g1_name} vs {g2_name}",
            "gene": gene,
            "status": "too_few_positive_cells",
            "threshold": threshold,
            f"{g1_name}_n_total": int(len(x1_all)),
            f"{g2_name}_n_total": int(len(x2_all)),
            f"{g1_name}_pct>thr": prev1,
            f"{g2_name}_pct>thr": prev2,
            f"{g1_name}_n_pos": int(len(x1_pos)),
            f"{g2_name}_n_pos": int(len(x2_pos)),
        }

    u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

    return {
        "pair": f"{g1_name} vs {g2_name}",
        "gene": gene,
        "status": "ok",
        "threshold": threshold,
        "U": float(u),
        "p_value": float(p),
        "median_pos_diff(g1-g2)": float(np.median(x1_pos) - np.median(x2_pos)),
        "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        f"{g1_name}_n_total": int(len(x1_all)),
        f"{g2_name}_n_total": int(len(x2_all)),
        f"{g1_name}_pct>thr": prev1,
        f"{g2_name}_pct>thr": prev2,
        f"{g1_name}_n_pos": int(len(x1_pos)),
        f"{g2_name}_n_pos": int(len(x2_pos)),
        f"{g1_name}_median_pos": float(np.median(x1_pos)),
        f"{g2_name}_median_pos": float(np.median(x2_pos)),
        f"{g1_name}_mean_pos": float(np.mean(x1_pos)),
        f"{g2_name}_mean_pos": float(np.mean(x2_pos)),
    }

def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def plot_positive_only(ad1, ad2, g1_name, g2_name, gene, threshold=0.0,
                       use_layer=None, use_raw=False, max_points=3000,
                       violin_gap=0.35, violin_width=0.28,
                       p_value=None):
    """
    只画表达细胞（>threshold）的分布
    额外功能：
      1) 缩窄两个violin间距：violin_gap
      2) x轴下方显示 mean_pos + pct>thr
      3) 两个violin上显示p值（若传入p_value）
    """
    x1_all = get_gene_vector(ad1, gene, use_layer=use_layer, use_raw=use_raw)
    x2_all = get_gene_vector(ad2, gene, use_layer=use_layer, use_raw=use_raw)
    if x1_all is None or x2_all is None:
        return

    x1_pos = x1_all[x1_all > threshold]
    x2_pos = x2_all[x2_all > threshold]

    # prevalence（all cells）
    pct1 = float(np.mean(x1_all > threshold)) * 100.0
    pct2 = float(np.mean(x2_all > threshold)) * 100.0

    # mean expression（positive-only，与你检验一致）
    mean1 = float(np.mean(x1_pos)) if len(x1_pos) > 0 else np.nan
    mean2 = float(np.mean(x2_pos)) if len(x2_pos) > 0 else np.nan

    # 位置：缩窄间距
    pos1 = 1.0
    pos2 = 1.0 + float(violin_gap)
    positions = [pos1, pos2]

    # 组装 data
    data = []
    labels = []
    if len(x1_pos) > 0:
        data.append(x1_pos); labels.append(g1_name)
    else:
        data.append(np.array([np.nan])); labels.append(g1_name)
    if len(x2_pos) > 0:
        data.append(x2_pos); labels.append(g2_name)
    else:
        data.append(np.array([np.nan])); labels.append(g2_name)

    fig, ax = plt.subplots(figsize=(4.5, 6))

    vp = ax.violinplot(
        data,
        positions=positions,
        showmeans=False,
        showmedians=True,
        showextrema=False,
        widths=violin_width,
        bw_method=0.3
    )

    def jitter(n, center):
        return center + (np.random.rand(n) - 0.5) * (violin_width * 0.7)

    def subsample(x):
        x = np.asarray(x)
        x = x[np.isfinite(x)]
        if len(x) > max_points:
            idx = np.random.choice(len(x), max_points, replace=False)
            return x[idx]
        return x

    # 点
    for center, x in zip(positions, data):
        xp = subsample(x)
        if len(xp) == 0:
            continue
        ax.scatter(jitter(len(xp), center), xp, s=4, alpha=0.25)

    # x轴标签：加 mean + pct
    label1 = f"{g1_name}\nmean expression={mean1:.3g}\nexpressing cells (%)={pct1:.1f}"
    label2 = f"{g2_name}\nmean expression={mean2:.3g}\nexpressing cells (%)={pct2:.1f}"
    ax.set_xticks(positions)
    ax.set_xticklabels([label1, label2], ha="center")

    ax.set_ylabel(f"Expression (only > {threshold})")
    ax.set_title(f"{gene}: {g1_name} vs {g2_name}")
    # 在violin上方显示p值（黑色横线）
    if p_value is not None and np.isfinite(p_value):

        # y 轴参考高度
        ymax = np.nanmax(
            np.concatenate([subsample(x1_pos), subsample(x2_pos)])
        ) if (len(x1_pos) + len(x2_pos)) > 0 else 1.0

        y_line = ymax + (0.08 * (ymax if ymax > 0 else 1.0))

        # ===== 黑色横线 =====
        ax.plot(
            [pos1, pos2],
            [y_line, y_line],
            color="black",
            lw=1.5
        )

        # p 值文本
        if p_value < 1e-3:
            ptxt = f"p={p_value:.1e}"
        else:
            ptxt = f"p={p_value:.3g}"

        ax.text(
            (pos1 + pos2) / 2,
            y_line + (0.02 * (ymax if ymax > 0 else 1.0)),
            ptxt,
            ha="center",
            va="bottom",
            color="black"
        )

        # 给上方留空间
        ax.set_ylim(top=y_line * 1.15)

    plt.tight_layout()
    plt.show()

# ===== 主流程：跑 conditional test + 画 positive-only 图 =====
USE_LAYER = None
USE_RAW = False
THRESHOLD = 0.0
MIN_POS_CELLS = 20

rows = []
for g1, g2 in pairs:
    ad1 = adatas[g1]
    ad2 = adatas[g2]
    for gene in genes:
        r = conditional_mw_positive_only(
            ad1, ad2, g1, g2, gene,
            min_pos_cells=MIN_POS_CELLS,
            threshold=THRESHOLD,
            use_layer=USE_LAYER,
            use_raw=USE_RAW
        )
        rows.append(r)

        # 画图：传入 p_value（若 ok），否则不显示 p
        if r.get("status") in ("ok", "too_few_positive_cells"):
            plot_positive_only(
                ad1, ad2, g1, g2, gene,
                threshold=THRESHOLD,
                use_layer=USE_LAYER,
                use_raw=USE_RAW,
                violin_gap=0.13,      # ✅ 间距更窄：你可调 0.2~0.5
                violin_width=0.1,    # ✅ violin 更瘦一些
                p_value=r.get("p_value", None) if r.get("status") == "ok" else None
            )

res_df = pd.DataFrame(rows)

# BH-FDR（只对 status==ok 的检验进行）
mask_ok = (res_df["status"] == "ok") & res_df.get("p_value", pd.Series([np.nan]*len(res_df))).notna()
if mask_ok.any():
    res_df.loc[mask_ok, "FDR_BH"] = bh_fdr(res_df.loc[mask_ok, "p_value"].values)

# 排序打印
cols_first = ["pair", "gene", "status", "p_value", "FDR_BH",
              "median_pos_diff(g1-g2)", "mean_pos_diff(g1-g2)",
              "threshold"]
cols_first = [c for c in cols_first if c in res_df.columns]
print(res_df.sort_values(["pair", "gene"])[cols_first + [c for c in res_df.columns if c not in cols_first]].to_string(index=False))

# 可选：保存
res_df.to_csv("conditional_positive_only_DE_two_genes.csv", index=False)


In [ ]:
mean_expr_dict = {}

for group_name, ad in adatas.items():
    # 保证基因都在 var_names 里
    genes_in_adata = [g for g in genes if g in ad.var_names]
    if len(genes_in_adata) == 0:
        continue

    # subset to genes
    ad_sub = ad[:, genes_in_adata]
    df = ad_sub.to_df()      # cells × genes

    # 每个基因在该组（所有细胞）的平均表达
    mean_expr = df.mean(axis=0)  # index=gene
    mean_expr_dict[group_name] = mean_expr

# 组合成一个大表：行=组，列=基因
mean_expr_df = pd.DataFrame(mean_expr_dict).T  # groups × genes
# 为了列顺序统一
mean_expr_df = mean_expr_df.reindex(columns=genes)

print(mean_expr_df.head())


In [ ]:
mean_expr_df.to_csv('Konopleva_BCL2L1expression.csv')